# Lab 01b: O Custo da Conversa (Overhead, Resiliência e Filas)

**Objetivo:** Implementar a lógica de coordenação de um cluster e medir como a latência de rede, as falhas de hardware e a **Teoria das Filas** impactam o SLA de sistemas bancários massivos.
**Ambiente:** Google Colab.
**Cenário:** Planejamento de infraestrutura para o sistema de processamento de transações PIX (CAIXA).

## 🛠️ Passo 1: O Simulador de Rede Distribuída

Em sistemas distribuídos, cada vez que enviamos uma tarefa para outro servidor, pagamos um preço de rede (Latência). Vamos criar uma função que simula essa "viagem" do dado e a chance de falha do servidor worker.

In [ ]:
import time
import random
import numpy as np
import matplotlib.pyplot as plt

def enviar_e_processar(batch_size, latencia_ms, taxa_falha=0.01):
    """
    Simula o envio de um lote de transações para um worker.
    latencia_ms: Custo fixo de ida e volta (Round Trip Time).
    taxa_falha: Probabilidade do servidor worker falhar (exige re-envio).
    """
    # Simula latência de rede (RTT)
    time.sleep(latencia_ms / 1000)
    
    # Simula falha aleatória de hardware (ex: nó reiniciou)
    if random.random() < taxa_falha:
        return None # Falhou!
    
    # Simula tempo de processamento das tarefas (0.001s por item)
    tempo_cpu = batch_size * 0.001
    time.sleep(tempo_cpu)
    
    return True # Sucesso!

## 🚀 Passo 2: O Desafio do Batch Size (Tamanho do Lote)

**Cenário:** Temos **1.000 transações** para processar com uma latência de rede de **100ms** (RTT).

**Exercício:** Implemente um loop que teste os tamanhos de lote `[1, 10, 50, 100, 500, 1000]` e meça o tempo total de execução.

In [ ]:
total_tarefas = 1000
latencia = 100 # ms
batch_sizes = [1, 10, 50, 100, 500, 1000]
tempos_execucao = []

for b_size in batch_sizes:
    print(f"Testando Lote de {b_size}...")
    start_t = time.time()
    
    processadas = 0
    while processadas < total_tarefas:
        if enviar_e_processar(b_size, latencia):
            processadas += b_size
        else:
            # Se falhar, tentamos novamente o mesmo lote
            pass 
            
    tempos_execucao.append(time.time() - start_t)

plt.figure(figsize=(10, 5))
plt.bar([str(b) for b in batch_sizes], tempos_execucao, color='orange')
plt.title('Impacto do Batch Size no Tempo Total (Latência 100ms)')
plt.ylabel('Segundos')
plt.xlabel('Tamanho do Lote')
plt.show()

### 🧠 Provocação 01:

**Pergunta:** Por que o lote de tamanho 1 foi o mais lento? Como o **Batching** ajuda a "esconder" a latência de rede?

## ⚡ Passo 3: O Paradoxo do Crescimento (Saturação)

Vamos simular um cluster crescendo de 1 a 100 máquinas. 
**Regra:** Adicionar máquinas diminui o tempo de cálculo, mas **aumenta o custo de coordenação** (Master/Worker sincronismo).

In [ ]:
def simular_cluster(n_nos, total_tarefas, latencia_ms):
    tempo_calculo = (total_tarefas * 0.001) / n_nos
    # Overhead de coordenação cresce linearmente com os nós
    overhead_rede = (latencia_ms / 1000) * n_nos
    return tempo_calculo + overhead_rede

nos = np.arange(1, 101)
resultados_cluster = [simular_cluster(n, 100000, 20) for n in nos] 

plt.figure(figsize=(12, 6))
plt.plot(nos, resultados_cluster, lw=3, color='blue')
plt.axvline(nos[np.argmin(resultados_cluster)], color='red', linestyle='--', label='Ponto de Saturação')
plt.title('Identificando o Limite de Expansão do Cluster')
plt.xlabel('Número de Máquinas')
plt.ylabel('Tempo Total (s)')
plt.legend(); plt.grid(True); plt.show()

print(f"O cluster para de ser eficiente após {nos[np.argmin(resultados_cluster)]} máquinas.")

### 🧠 Provocação 02:

**Pergunta:** Observe o gráfico acima. Como você explicaria para o banco que "menos máquinas" pode ser melhor que "mais máquinas"?

## 🚦 Passo 4: Teoria das Filas (O Gargalo Invisível)

Na vida real, as transações chegam em um fluxo constante. Se a taxa de chegada ($\lambda$) for maior que a capacidade de processamento do cluster ($\mu$), a fila cresce infinitamente.

In [ ]:
def tempo_espera_fila(chegada, capacidade):
    if chegada >= capacidade:
        return 500 # Simula colapso
    return 1 / (capacidade - chegada)

taxas_chegada = np.linspace(10, 99, 100)
capacidade_cluster = 100
esperas = [tempo_espera_fila(c, capacidade_cluster) for c in taxas_chegada]

plt.figure(figsize=(10, 5))
plt.plot(taxas_chegada, esperas, color='purple', lw=2)
plt.title('Teoria das Filas: O "Joelho" da Curva')
plt.xlabel('Carga do Sistema (Requisições/Seg)')
plt.ylabel('Tempo de Espera na Fila (s)')
plt.grid(True); plt.show()

### 🧠 Provocação 03:

**Pergunta:** Por que o tempo de espera explode violentamente quando passamos de 90% de ocupação? O que acontece com o PIX da CAIXA se o cluster operar no limite?

## 🔥 Desafio Extra (Tier 2)

### Resiliência e Exponential Backoff
Implemente uma lógica de **Retry** onde o tempo de espera dobra a cada falha sucessiva (Backoff) e compare o tempo final.

## 🏁 Conclusão de Engenharia

Neste laboratório, vimos que:
1. **Batching:** É a ferramenta número 1 para esconder a latência.
2. **Limites Físicos:** O custo de coordenação limita a escala horizontal.
3. **SLA:** Nunca dimensione um sistema para rodar a 100% de capacidade.